# Notebook 16 — Transition Operator, Entropy, and Mixing

**Repo:** `prime-numbers-lab`  
**Notebook path:** `notebooks/16_transition_operator_entropy_mixing.ipynb`

This notebook extends Notebook 15 from residue-class residual structure into a transition-operator view.

Core question:

> If prime gaps are globally close to an Exp(1)-like normalized distribution, what local transition structure remains between prime residue classes?

We model prime residue transitions modulo 30:

\[
r_n = p_n \bmod 30,\qquad r_{n+1} = p_{n+1} \bmod 30
\]

and estimate:

\[
P(r_{n+1}=j \mid r_n=i)
\]

Then we evaluate transition matrix structure, stationary distribution, entropy by anchor residue, mixing behavior, spectral gap proxy, residual transition mass, scale-window transition drift, and interpretation numbers.

In [ ]:
import os
import math
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except Exception:
    display = None
    Markdown = None

NOTEBOOK_ID = "16"
NOTEBOOK_TITLE = "transition_operator_entropy_mixing"
RANDOM_SEED = 9423

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PRIME_LIMIT = 2_000_000
RESIDUES30 = np.array([1, 7, 11, 13, 17, 19, 23, 29], dtype=int)
RESIDUE_LABELS = [str(r) for r in RESIDUES30]

print(f"Notebook {NOTEBOOK_ID}: {NOTEBOOK_TITLE}")
print(f"Prime limit: {PRIME_LIMIT:,}")
print(f"Outputs: {OUTPUT_DIR.resolve()}")

## 1. Prime generation

Self-contained sieve of Eratosthenes, no external data.

In [ ]:
def primes_upto(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=np.int64)
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False
    sieve[4::2] = False
    root = int(math.isqrt(n))
    for p in range(3, root + 1, 2):
        if sieve[p]:
            sieve[p*p::2*p] = False
    return np.flatnonzero(sieve).astype(np.int64)

primes = primes_upto(PRIME_LIMIT)
primes = primes[primes > 5]

gaps = np.diff(primes)
anchor_primes = primes[:-1]
next_primes = primes[1:]
anchor_residue = anchor_primes % 30
next_residue = next_primes % 30
gap_residue = gaps % 30
z = gaps / np.log(anchor_primes)

valid = np.isin(anchor_residue, RESIDUES30) & np.isin(next_residue, RESIDUES30)
anchor_primes = anchor_primes[valid]
next_primes = next_primes[valid]
gaps = gaps[valid]
z = z[valid]
anchor_residue = anchor_residue[valid]
next_residue = next_residue[valid]
gap_residue = gap_residue[valid]

print("Prime count:", len(primes))
print("Transition count:", len(anchor_residue))
print("Mean normalized gap z:", float(np.mean(z)))
print("Std normalized gap z:", float(np.std(z)))

## 2. Transition matrix

\[
P_{ij}=\Pr(r_{n+1}=j\mid r_n=i)
\]

where \(i,j\in\{1,7,11,13,17,19,23,29\}\).

In [ ]:
residue_to_index = {r: i for i, r in enumerate(RESIDUES30)}
n_res = len(RESIDUES30)

counts = np.zeros((n_res, n_res), dtype=float)
for a, b in zip(anchor_residue, next_residue):
    counts[residue_to_index[int(a)], residue_to_index[int(b)]] += 1

row_sums = counts.sum(axis=1, keepdims=True)
P = np.divide(counts, row_sums, out=np.zeros_like(counts), where=row_sums > 0)

transition_df = pd.DataFrame(P, index=RESIDUE_LABELS, columns=RESIDUE_LABELS)
transition_counts_df = pd.DataFrame(counts.astype(int), index=RESIDUE_LABELS, columns=RESIDUE_LABELS)

display(transition_df.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(P, aspect="auto")
ax.set_title("Prime residue transition operator mod30")
ax.set_xlabel("next residue p_{n+1} mod 30")
ax.set_ylabel("anchor residue p_n mod 30")
ax.set_xticks(np.arange(n_res))
ax.set_yticks(np.arange(n_res))
ax.set_xticklabels(RESIDUE_LABELS)
ax.set_yticklabels(RESIDUE_LABELS)
plt.colorbar(im, ax=ax, label="transition probability")
for i in range(n_res):
    for j in range(n_res):
        ax.text(j, i, f"{P[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout()
fig.savefig(FIG_DIR / "16_transition_operator_heatmap.png", dpi=160)
plt.show()

## 3. Stationary distribution

\[
\pi P=\pi
\]

If residue transitions mix toward uniformity, \(\pi\) should be close to \(1/8\) per residue.

In [ ]:
def stationary_distribution(P: np.ndarray) -> np.ndarray:
    evals, evecs = np.linalg.eig(P.T)
    idx = np.argmin(np.abs(evals - 1))
    v = np.real(evecs[:, idx])
    v = np.maximum(v, 0)
    if v.sum() == 0:
        v = np.abs(np.real(evecs[:, idx]))
    return v / v.sum()

pi = stationary_distribution(P)
uniform = np.ones(n_res) / n_res

stationary_df = pd.DataFrame({
    "residue": RESIDUES30,
    "stationary_pi": pi,
    "uniform": uniform,
    "delta_from_uniform": pi - uniform,
})
display(stationary_df.round(6))

stationary_l1 = float(np.sum(np.abs(pi - uniform)))
stationary_l2 = float(np.linalg.norm(pi - uniform))
print("Stationary L1 distance to uniform:", stationary_l1)
print("Stationary L2 distance to uniform:", stationary_l2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.bar(RESIDUE_LABELS, pi, label="stationary distribution")
ax.axhline(1/n_res, linestyle="--", label="uniform 1/8")
ax.set_title("Stationary distribution of mod30 transition operator")
ax.set_xlabel("residue")
ax.set_ylabel("probability")
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
fig.savefig(FIG_DIR / "16_stationary_distribution.png", dpi=160)
plt.show()

## 4. Entropy by anchor residue

\[
H_i=-\frac{1}{\log 8}\sum_j P_{ij}\log P_{ij}
\]

In [ ]:
def normalized_entropy(p: np.ndarray) -> float:
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    if len(p) == 0:
        return 0.0
    return float(-np.sum(p * np.log(p)) / np.log(n_res))

row_entropy = np.array([normalized_entropy(P[i]) for i in range(n_res)])
row_max = P.max(axis=1)
row_argmax = RESIDUES30[np.argmax(P, axis=1)]

entropy_df = pd.DataFrame({
    "anchor_residue": RESIDUES30,
    "normalized_entropy": row_entropy,
    "max_transition_probability": row_max,
    "most_likely_next_residue": row_argmax,
})
display(entropy_df.round(6))
print("Mean normalized transition entropy:", float(row_entropy.mean()))
print("Min normalized transition entropy:", float(row_entropy.min()))
print("Max normalized transition entropy:", float(row_entropy.max()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(RESIDUE_LABELS, row_entropy, marker="o", label="normalized row entropy")
ax.axhline(1.0, linestyle="--", label="maximum entropy")
ax.set_title("Transition entropy by anchor residue mod30")
ax.set_xlabel("anchor residue p_n mod 30")
ax.set_ylabel("normalized entropy")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
fig.savefig(FIG_DIR / "16_transition_entropy_by_anchor.png", dpi=160)
plt.show()

## 5. Mixing through powers of \(P\)

We measure how quickly \(P^k\) rows approach stationary distribution \(\pi\).

In [ ]:
def tv_distance(p, q):
    return 0.5 * np.sum(np.abs(np.asarray(p) - np.asarray(q)))

max_k = 20
mixing_rows = []
Pk = np.eye(n_res)

for k in range(1, max_k + 1):
    Pk = Pk @ P
    tv_by_row = np.array([tv_distance(Pk[i], pi) for i in range(n_res)])
    l2_by_row = np.array([np.linalg.norm(Pk[i] - pi) for i in range(n_res)])
    mixing_rows.append({
        "k": k,
        "max_tv_to_stationary": float(tv_by_row.max()),
        "mean_tv_to_stationary": float(tv_by_row.mean()),
        "max_l2_to_stationary": float(l2_by_row.max()),
        "mean_l2_to_stationary": float(l2_by_row.mean()),
    })

mixing_df = pd.DataFrame(mixing_rows)
display(mixing_df.round(6).head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(mixing_df["k"], mixing_df["max_tv_to_stationary"], marker="o", label="max TV distance")
ax.plot(mixing_df["k"], mixing_df["mean_tv_to_stationary"], marker="o", label="mean TV distance")
ax.set_title("Mixing toward stationary distribution")
ax.set_xlabel("transition steps k")
ax.set_ylabel("total variation distance")
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
fig.savefig(FIG_DIR / "16_mixing_distance_vs_steps.png", dpi=160)
plt.show()

## 6. Spectral gap proxy

\[
\lambda_2=\max_{i\ne1}|\lambda_i|,\qquad \text{gap}=1-\lambda_2
\]

In [ ]:
evals = np.linalg.eigvals(P)
abs_evals = np.sort(np.abs(evals))[::-1]
lambda_1 = float(abs_evals[0])
lambda_2 = float(abs_evals[1])
spectral_gap = float(1 - lambda_2)

spectral_df = pd.DataFrame({
    "metric": ["lambda_1_abs", "lambda_2_abs", "spectral_gap_proxy"],
    "value": [lambda_1, lambda_2, spectral_gap],
})
display(spectral_df.round(8))

print(f"Second eigenvalue magnitude lambda2 ≈ {lambda_2:.6f}")
print(f"Spectral gap proxy 1-lambda2 ≈ {spectral_gap:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.bar([str(i) for i in range(len(abs_evals))], abs_evals)
ax.set_title("Transition operator eigenvalue magnitudes")
ax.set_xlabel("eigenvalue rank")
ax.set_ylabel("|lambda|")
ax.grid(True, alpha=0.35)
plt.tight_layout()
fig.savefig(FIG_DIR / "16_transition_eigenvalue_magnitudes.png", dpi=160)
plt.show()

## 7. Transition residual mass

For each transition \(r_n\to r_{n+1}\), compare normalized-gap density against Exp(1).

In [ ]:
def exp_pdf(x):
    return np.exp(-x)

def density_curve(values, bins):
    hist, edges = np.histogram(values, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, hist

z_bins = np.linspace(0, 6, 61)
z_centers = 0.5 * (z_bins[:-1] + z_bins[1:])
exp_ref = exp_pdf(z_centers)

transition_metrics = []

for a in RESIDUES30:
    for b in RESIDUES30:
        mask = (anchor_residue == a) & (next_residue == b)
        vals = z[mask]
        if len(vals) < 20:
            continue
        _, dens = density_curve(vals, z_bins)
        delta = dens - exp_ref
        l1 = float(np.mean(np.abs(delta)))
        l2 = float(np.sqrt(np.mean(delta**2)))
        transition_metrics.append({
            "transition": f"{a}->{b}",
            "anchor": int(a),
            "next": int(b),
            "count": int(len(vals)),
            "probability": float(P[residue_to_index[int(a)], residue_to_index[int(b)]]),
            "mean_z": float(np.mean(vals)),
            "l1_residual_mass": l1,
            "l2_residual_mass": l2,
            "tail_p_z_gt_2": float(np.mean(vals > 2)),
            "tail_p_z_gt_3": float(np.mean(vals > 3)),
        })

transition_metrics_df = pd.DataFrame(transition_metrics)
transition_metrics_df = transition_metrics_df.sort_values("l1_residual_mass", ascending=False).reset_index(drop=True)
display(transition_metrics_df.head(20).round(6))

In [ ]:
top = transition_metrics_df.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(top["transition"], top["l1_residual_mass"])
ax.set_title("Top transition residual masses mod30")
ax.set_xlabel("L1 residual mass vs Exp(1)")
ax.set_ylabel("transition p_n mod30 -> p_{n+1} mod30")
ax.grid(True, alpha=0.35)
plt.tight_layout()
fig.savefig(FIG_DIR / "16_top_transition_residual_masses.png", dpi=160)
plt.show()

## 8. Transition probability lift

\[
\text{lift}_{ij}=\frac{P_{ij}}{\Pr(r_{n+1}=j)}
\]

In [ ]:
next_probs = np.array([np.mean(next_residue == r) for r in RESIDUES30])
lift = np.divide(P, next_probs.reshape(1, -1), out=np.zeros_like(P), where=next_probs.reshape(1, -1) > 0)

lift_df = pd.DataFrame(lift, index=RESIDUE_LABELS, columns=RESIDUE_LABELS)
display(lift_df.round(4))

lift_records = []
for i, a in enumerate(RESIDUES30):
    for j, b in enumerate(RESIDUES30):
        lift_records.append({
            "transition": f"{a}->{b}",
            "anchor": int(a),
            "next": int(b),
            "lift": float(lift[i, j]),
            "probability": float(P[i, j]),
            "count": int(counts[i, j]),
        })
lift_records_df = pd.DataFrame(lift_records).sort_values("lift", ascending=False)
display(lift_records_df.head(20).round(6))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(lift, aspect="auto")
ax.set_title("Transition lift over independent next-residue baseline")
ax.set_xlabel("next residue p_{n+1} mod 30")
ax.set_ylabel("anchor residue p_n mod 30")
ax.set_xticks(np.arange(n_res))
ax.set_yticks(np.arange(n_res))
ax.set_xticklabels(RESIDUE_LABELS)
ax.set_yticklabels(RESIDUE_LABELS)
plt.colorbar(im, ax=ax, label="lift ratio")
for i in range(n_res):
    for j in range(n_res):
        ax.text(j, i, f"{lift[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout()
fig.savefig(FIG_DIR / "16_transition_lift_heatmap.png", dpi=160)
plt.show()

## 9. Windowed transition drift

For each log-spaced prime window:

\[
\|P_x-P\|_1
\]

In [ ]:
n_windows = 14
x_edges = np.geomspace(anchor_primes.min(), anchor_primes.max(), n_windows + 1)
window_rows = []

for w in range(n_windows):
    lo, hi = x_edges[w], x_edges[w + 1]
    mask = (anchor_primes >= lo) & (anchor_primes < hi)
    if mask.sum() < 100:
        continue

    Cw = np.zeros((n_res, n_res), dtype=float)
    for a, b in zip(anchor_residue[mask], next_residue[mask]):
        Cw[residue_to_index[int(a)], residue_to_index[int(b)]] += 1

    rs = Cw.sum(axis=1, keepdims=True)
    Pw = np.divide(Cw, rs, out=np.zeros_like(Cw), where=rs > 0)
    dist_l1 = float(np.mean(np.abs(Pw - P)))
    ent = np.array([normalized_entropy(Pw[i]) for i in range(n_res)])
    window_rows.append({
        "window": w,
        "x_lo": float(lo),
        "x_hi": float(hi),
        "x_mid": float(math.sqrt(lo * hi)),
        "n": int(mask.sum()),
        "mean_transition_entropy": float(np.mean(ent)),
        "min_transition_entropy": float(np.min(ent)),
        "transition_operator_l1_drift": dist_l1,
    })

window_df = pd.DataFrame(window_rows)
display(window_df.round(6))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(window_df["x_mid"], window_df["transition_operator_l1_drift"], marker="o", label="operator L1 drift")
ax.set_xscale("log")
ax.set_title("Windowed transition-operator drift vs scale")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("mean |P_window - P_global|")
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
fig.savefig(FIG_DIR / "16_windowed_transition_operator_drift.png", dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(window_df["x_mid"], window_df["mean_transition_entropy"], marker="o", label="mean entropy")
ax.plot(window_df["x_mid"], window_df["min_transition_entropy"], marker="o", label="min row entropy")
ax.axhline(1.0, linestyle="--", label="maximum entropy")
ax.set_xscale("log")
ax.set_title("Windowed transition entropy vs scale")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("normalized entropy")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
fig.savefig(FIG_DIR / "16_windowed_transition_entropy.png", dpi=160)
plt.show()

## 10. Residual-weighted transition operator

In [ ]:
resid_matrix = np.zeros((n_res, n_res), dtype=float)

metric_lookup = {
    (int(row.anchor), int(row.next)): float(row.l1_residual_mass)
    for _, row in transition_metrics_df.iterrows()
}

for i, a in enumerate(RESIDUES30):
    for j, b in enumerate(RESIDUES30):
        resid_matrix[i, j] = metric_lookup.get((int(a), int(b)), 0.0)

weighted_residual = P * resid_matrix
total_wr = weighted_residual.sum()
weighted_residual_share = weighted_residual / total_wr if total_wr > 0 else weighted_residual

weighted_residual_df = pd.DataFrame(weighted_residual_share, index=RESIDUE_LABELS, columns=RESIDUE_LABELS)
display(weighted_residual_df.round(5))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(weighted_residual_share, aspect="auto")
ax.set_title("Residual-weighted transition contribution")
ax.set_xlabel("next residue p_{n+1} mod 30")
ax.set_ylabel("anchor residue p_n mod 30")
ax.set_xticks(np.arange(n_res))
ax.set_yticks(np.arange(n_res))
ax.set_xticklabels(RESIDUE_LABELS)
ax.set_yticklabels(RESIDUE_LABELS)
plt.colorbar(im, ax=ax, label="share of residual-weighted transition mass")
for i in range(n_res):
    for j in range(n_res):
        ax.text(j, i, f"{weighted_residual_share[i,j]:.3f}", ha="center", va="center", fontsize=7)
plt.tight_layout()
fig.savefig(FIG_DIR / "16_residual_weighted_transition_operator.png", dpi=160)
plt.show()

## 11. Transition-conditioned residual curves

\[
\Delta_{i\to j}(z)=f_{i\to j}(z)-e^{-z}
\]

In [ ]:
top_transitions = transition_metrics_df.head(6)["transition"].tolist()

fig, ax = plt.subplots(figsize=(11, 7))
for tr in top_transitions:
    a, b = [int(x) for x in tr.split("->")]
    mask = (anchor_residue == a) & (next_residue == b)
    vals = z[mask]
    centers, dens = density_curve(vals, z_bins)
    delta = dens - exp_ref
    ax.plot(centers, delta, label=tr)

ax.axhline(0, linestyle="--")
ax.set_title("Strongest transition-conditioned residual curves")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("Delta(z) = empirical PDF - Exp(1)")
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
fig.savefig(FIG_DIR / "16_transition_conditioned_residual_curves.png", dpi=160)
plt.show()

## 12. Interpretation numbers

In [ ]:
top_lift = lift_records_df.iloc[0]
top_resid = transition_metrics_df.iloc[0]
mean_entropy = float(row_entropy.mean())
min_entropy_row = entropy_df.sort_values("normalized_entropy").iloc[0]
max_entropy_row = entropy_df.sort_values("normalized_entropy", ascending=False).iloc[0]

summary = {
    "prime_limit": PRIME_LIMIT,
    "transition_count": int(len(anchor_residue)),
    "stationary_l1_to_uniform": stationary_l1,
    "stationary_l2_to_uniform": stationary_l2,
    "mean_row_entropy": mean_entropy,
    "min_entropy_anchor_residue": int(min_entropy_row["anchor_residue"]),
    "min_entropy_value": float(min_entropy_row["normalized_entropy"]),
    "max_entropy_anchor_residue": int(max_entropy_row["anchor_residue"]),
    "max_entropy_value": float(max_entropy_row["normalized_entropy"]),
    "lambda2_abs": lambda_2,
    "spectral_gap_proxy": spectral_gap,
    "top_lift_transition": str(top_lift["transition"]),
    "top_lift_value": float(top_lift["lift"]),
    "top_residual_transition": str(top_resid["transition"]),
    "top_residual_l1_mass": float(top_resid["l1_residual_mass"]),
    "windowed_operator_drift_min": float(window_df["transition_operator_l1_drift"].min()),
    "windowed_operator_drift_max": float(window_df["transition_operator_l1_drift"].max()),
}

print(json.dumps(summary, indent=2))

interpretation_text = f"""
### Notebook 16 interpretation

- Transition count: **{summary['transition_count']:,}**
- Stationary distribution distance to uniform: **L1={summary['stationary_l1_to_uniform']:.6f}**
- Mean normalized transition entropy: **{summary['mean_row_entropy']:.6f}**
- Minimum row entropy: residue **{summary['min_entropy_anchor_residue']}** with **{summary['min_entropy_value']:.6f}**
- Spectral gap proxy: **{summary['spectral_gap_proxy']:.6f}**
- Strongest lift transition: **{summary['top_lift_transition']}**, lift **{summary['top_lift_value']:.6f}**
- Strongest transition residual: **{summary['top_residual_transition']}**, L1 residual mass **{summary['top_residual_l1_mass']:.6f}**
- Windowed transition-operator drift range: **{summary['windowed_operator_drift_min']:.6f} → {summary['windowed_operator_drift_max']:.6f}**

Core result:

> Prime residue transitions mod30 form a high-entropy, near-mixing finite transition operator, while specific residue transitions still carry measurable residual structure relative to the global Exp(1)-like normalized gap baseline.
"""

if Markdown is not None:
    display(Markdown(interpretation_text))
else:
    print(interpretation_text)

## 13. Export outputs

In [ ]:
transition_df.to_csv(OUTPUT_DIR / "16_transition_operator.csv")
transition_counts_df.to_csv(OUTPUT_DIR / "16_transition_counts.csv", index=True)
stationary_df.to_csv(OUTPUT_DIR / "16_stationary_distribution.csv", index=False)
entropy_df.to_csv(OUTPUT_DIR / "16_transition_entropy_by_anchor.csv", index=False)
mixing_df.to_csv(OUTPUT_DIR / "16_mixing_metrics.csv", index=False)
spectral_df.to_csv(OUTPUT_DIR / "16_spectral_metrics.csv", index=False)
transition_metrics_df.to_csv(OUTPUT_DIR / "16_transition_residual_metrics.csv", index=False)
lift_records_df.to_csv(OUTPUT_DIR / "16_transition_lift_records.csv", index=False)
window_df.to_csv(OUTPUT_DIR / "16_windowed_transition_drift.csv", index=False)
weighted_residual_df.to_csv(OUTPUT_DIR / "16_residual_weighted_transition_operator.csv")

with open(OUTPUT_DIR / "16_interpretation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Exported Notebook 16 outputs.")
print("CSV/JSON output directory:", OUTPUT_DIR.resolve())
print("Figure directory:", FIG_DIR.resolve())

## 14. Paper-ready takeaway

Notebook 16 formalizes the transition layer suggested by Notebook 15:

\[
P(r_{n+1}\mid r_n)
\]

The result is not that modular residues destroy the exponential gap model. Instead:

1. normalized gaps remain globally Exp(1)-like,
2. scale-window residuals shrink,
3. residue classes preserve measurable local structure,
4. transition operators encode this local structure as a near-mixing but nontrivial finite Markov system.

Paper phrasing:

> The prime-gap baseline is distributionally stable, while the residue-transition operator retains arithmetic memory. This memory appears as high-entropy but nonuniform transition lift, residual-weighted transition mass, and transition-conditioned deviations from the Exp(1) normalized-gap baseline.